# RQ2 — CV-HIL Component Ablation (A–D)

Trả lời **RQ2**: candidate retrieval, cross-model verification, và taxonomy-aware validation đóng góp gì vào chất lượng nhãn?

| Variant | Retrieval | Verification | Validation | LLM calls |
|---------|-----------|--------------|------------|-----------|
| **A** `cvhil_a` | off | off | off | annotate |
| **B** `cvhil_b` | on | off | off | annotate |
| **C** `cvhil_c` | on | on | off | annotate + verify |
| **D** `cvhil_d` | on | on | on | **reuse** `verified_d2` |

**Scope mặc định:** D3 human gold (300 mẫu). Đánh giá cùng metric RQ1 (`eval_predictions_vs_d3`).

## Files cần trên Drive / local
```
data/gold/d3_dataset.json
data/annotations/pool.json
data/annotations/verified_d2.json          # variant D
data/code/labeling_pipeline.py
experiments/annotation_benchmark/eval_rq2_ablation.py
experiments/annotation_benchmark/common.py
```

**Env:** `OPENAI_API_KEY`, `MONGODB_URI` (B/C cần retrieval; A chỉ cần OpenAI).

## Workflow
1. §1 Config → §2 Import pipeline → §3 Smoke (3 mẫu) → §4 Batch A/B/C → §5 Build D → §6 Eval table

## 1. Config + paths

In [1]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_DRIVE_REPO = "/content/drive/MyDrive/intent_kb"

if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = Path(DEFAULT_DRIVE_REPO)
else:
    REPO_ROOT = Path("..").resolve()

os.chdir(REPO_ROOT)
os.environ["INTENT_REPO"] = str(REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "experiments" / "annotation_benchmark"))
sys.path.insert(0, str(REPO_ROOT / "data" / "code"))

ANNOTATIONS = REPO_ROOT / "data/annotations"
D3_FILE = REPO_ROOT / "data/gold/d3_dataset.json"
POOL_FILE = REPO_ROOT / "data/annotations/pool.json"
VERIFIED_D2 = ANNOTATIONS / "verified_d2.json"
OUTDIR = REPO_ROOT / "experiments/annotation_benchmark/results/rq2_ablation"
OUTDIR.mkdir(parents=True, exist_ok=True)

# --- Run controls ---
SCOPE = "d3"          # "d3" | "smoke"
SMOKE_N = 3
RUN_VARIANTS = ["cvhil_a", "cvhil_b", "cvhil_c"]  # D built from cache
FORCE_VERIFY = True   # fair C vs D: verify every sample when enable_verifier
SLEEP_SEC = 0.5       # rate limit between LLM calls
CHECKPOINT_EVERY = 10

VARIANTS = {
    "cvhil_a": dict(use_retrieval=False, enable_verifier=False, enable_validation=False),
    "cvhil_b": dict(use_retrieval=True,  enable_verifier=False, enable_validation=False),
    "cvhil_c": dict(use_retrieval=True,  enable_verifier=True,  enable_validation=False),
    "cvhil_d": dict(reuse_from="verified_d2"),
}

print("REPO_ROOT:", REPO_ROOT)
print("D3 exists:", D3_FILE.exists())
print("verified_d2 exists:", VERIFIED_D2.exists())
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))
print("MONGODB_URI set:", bool(os.environ.get("MONGODB_URI")))

Mounted at /content/drive
REPO_ROOT: /content/drive/MyDrive/intent_kb
D3 exists: True
verified_d2 exists: True
OPENAI_API_KEY set: False
MONGODB_URI set: False


## 2. Import production pipeline

Colab: set secrets trước khi chạy cell này (`OPENAI_API_KEY`, `MONGODB_URI`).

```python
import os
os.environ["OPENAI_API_KEY"] = "..."
os.environ["MONGODB_URI"] = "..."
os.environ["VERIFY_ALL_SAMPLES"] = "0"  # notebook dùng force_verify per-call
```

In [2]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.7 MB/s eta 0:00:00


In [3]:
from common import load_json, save_json, load_id_set, eval_predictions_vs_d3, load_d3_gold

import labeling_pipeline as lp
from labeling_pipeline import (
    annotate_cvhil_ablation,
    CVHIL_ABLATION_VARIANTS,
    enrich_sample,
)

print("Pipeline OK | variants:", list(CVHIL_ABLATION_VARIANTS))

MONGODB_URI (dan vao roi Enter): ··········
OPENAI_API_KEY (dan vao roi Enter): ··········
HASAKI_JSON: True /content/drive/MyDrive/intent_kb/data/hasaki_prelabel.json
OUTPUT_DIR : /content/drive/MyDrive/intent_kb/data/outputs
MONGODB_URI set: True
LLM model: gpt-4o-mini
Verifier: ON | model: gpt-4o | verify all: False | verify if conf < 0.85
Human review statuses: ['needs_review', 'pending_new_label_review']
OK db: intent_kb
intent_nodes: 653
intent_edges: 681
✅ OpenAI API model: gpt-4o-mini
Helpers OK
Pipeline OK | variants: ['cvhil_a', 'cvhil_b', 'cvhil_c', 'cvhil_d']


## 3. Helpers — load scope, checkpoint, build D

In [4]:
def load_scope_samples(scope="d3", smoke_n=3):
    d3_doc = load_json(D3_FILE)
    d3_rows = d3_doc["results"] if isinstance(d3_doc, dict) else d3_doc
    pool_doc = load_json(POOL_FILE)
    pool_rows = pool_doc["results"] if isinstance(pool_doc, dict) else pool_doc
    pool_by_id = {r["sample_id"]: r for r in pool_rows}

    samples = []
    for row in d3_rows:
        sid = row["sample_id"]
        base = dict(pool_by_id.get(sid, row))
        base["sample_id"] = sid
        base.setdefault("sentence", row.get("sentence", ""))
        samples.append(enrich_sample(base))

    if scope == "smoke":
        return samples[:smoke_n]
    return samples


def variant_output_path(variant_key: str) -> Path:
    return ANNOTATIONS / f"{variant_key}.json"


def load_checkpoint(variant_key: str) -> dict:
    path = variant_output_path(variant_key)
    if not path.exists():
        return {}
    doc = load_json(path)
    rows = doc.get("results", [])
    return {r["sample_id"]: r for r in rows}


def save_variant_run(variant_key: str, results: list, meta: dict | None = None):
    cfg = CVHIL_ABLATION_VARIANTS.get(variant_key, {})
    payload = {
        "method": variant_key,
        "display_name": cfg.get("display_name", variant_key),
        "num_samples": len(results),
        **(meta or {}),
        "results": results,
    }
    save_json(variant_output_path(variant_key), payload)


def build_cvhil_d_from_verified(scope="d3", smoke_n=3):
    if not VERIFIED_D2.exists():
        raise FileNotFoundError(f"Missing {VERIFIED_D2}")
    scope_ids = {s["sample_id"] for s in load_scope_samples(scope, smoke_n)}
    doc = load_json(VERIFIED_D2)
    rows = []
    for r in doc.get("results", []):
        if r["sample_id"] not in scope_ids:
            continue
        meta = dict(r.get("metadata") or {})
        meta.update({
            "variant": "cvhil_d",
            "use_retrieval": True,
            "enable_verifier": True,
            "enable_validation": True,
            "source": meta.get("source", "verified_d2_reuse"),
        })
        rows.append({
            "sample_id": r["sample_id"],
            "sentence": r.get("sentence", ""),
            "predicted_label": dict(r.get("predicted_label") or {}),
            "confidence": r.get("confidence", 0.0),
            "api_calls": 0,
            "input_tokens": 0,
            "output_tokens": 0,
            "cost_usd": 0.0,
            "metadata": meta,
        })
    save_variant_run(
        "cvhil_d",
        rows,
        meta={"reuse_from": "verified_d2", "scope": scope, "num_matched": len(rows)},
    )
    print(f"Built cvhil_d: {len(rows)} rows -> {variant_output_path('cvhil_d')}")
    return rows


def run_variant_batch(variant_key: str, samples: list):
    cfg = VARIANTS[variant_key]
    if cfg.get("reuse_from"):
        raise ValueError(f"{variant_key} is cache-only; use build_cvhil_d_from_verified()")

    done = load_checkpoint(variant_key)
    results = list(done.values())
    t0 = time.time()
    n_new = 0

    for i, sample in enumerate(samples):
        sid = sample["sample_id"]
        if sid in done:
            continue
        row = annotate_cvhil_ablation(
            sample,
            variant=variant_key,
            use_retrieval=cfg["use_retrieval"],
            enable_verifier=cfg["enable_verifier"],
            enable_validation=cfg["enable_validation"],
            force_verify=FORCE_VERIFY,
        )
        results.append(row)
        done[sid] = row
        n_new += 1
        if n_new % CHECKPOINT_EVERY == 0:
            save_variant_run(variant_key, results, meta={"checkpoint": True})
            print(f"  checkpoint {variant_key}: {len(results)}/{len(samples)}")
        if SLEEP_SEC:
            time.sleep(SLEEP_SEC)

    save_variant_run(
        variant_key,
        results,
        meta={"elapsed_seconds": time.time() - t0, "scope": SCOPE, "n_new": n_new},
    )
    print(f"Done {variant_key}: {len(results)} rows ({n_new} new)")
    return results


samples = load_scope_samples(SCOPE, SMOKE_N)
print(f"Scope={SCOPE} samples={len(samples)}")

Scope=d3 samples=300


## 4. Smoke test (3 mẫu × A/B/C)

Đặt `SCOPE="smoke"` ở §1 trước khi chạy.

In [5]:
SCOPE = "smoke"
samples = load_scope_samples(SCOPE, SMOKE_N)

for vk in ["cvhil_a", "cvhil_b", "cvhil_c"]:
    run_variant_batch(vk, samples)

build_cvhil_d_from_verified(SCOPE, SMOKE_N)

for vk in ["cvhil_a", "cvhil_b", "cvhil_c", "cvhil_d"]:
    doc = load_json(variant_output_path(vk))
    print(vk, "rows:", len(doc.get("results", [])))

Done cvhil_a: 300 rows (0 new)
Done cvhil_b: 300 rows (0 new)
Done cvhil_c: 183 rows (0 new)
Built cvhil_d: 3 rows -> /content/drive/MyDrive/intent_kb/data/annotations/cvhil_d.json
cvhil_a rows: 300
cvhil_b rows: 300
cvhil_c rows: 183
cvhil_d rows: 3


## 5. Full D3 batch (300 mẫu)

Đặt `SCOPE="d3"`. Resume tự động từ checkpoint.

In [7]:
SCOPE = "d3"
samples = load_scope_samples(SCOPE, SMOKE_N)
print(f"Running full scope: {len(samples)} samples")

for vk in RUN_VARIANTS:
    print("\n===", vk, "===")
    run_variant_batch(vk, samples)

build_cvhil_d_from_verified(SCOPE, SMOKE_N)

Running full scope: 300 samples

=== cvhil_a ===
Done cvhil_a: 300 rows (0 new)

=== cvhil_b ===
Done cvhil_b: 300 rows (0 new)

=== cvhil_c ===
  checkpoint cvhil_c: 193/300
  checkpoint cvhil_c: 203/300
  checkpoint cvhil_c: 213/300
  checkpoint cvhil_c: 223/300
  checkpoint cvhil_c: 233/300
  checkpoint cvhil_c: 243/300
  checkpoint cvhil_c: 253/300
  checkpoint cvhil_c: 263/300
  checkpoint cvhil_c: 273/300
  checkpoint cvhil_c: 283/300
  checkpoint cvhil_c: 293/300
Done cvhil_c: 300 rows (117 new)
Built cvhil_d: 300 rows -> /content/drive/MyDrive/intent_kb/data/annotations/cvhil_d.json


[{'sample_id': 'sua-rua-mat-cerave-sach-sau-cho-da-thuong-den-da-dau-473ml-102959_q02',
  'sentence': 'Vậy là sao nv bên bạn nghĩ mih k bít gì hay sao, mih k đôg ý như vậy. Chặn quà của khách hàng lại là sao. sao uy tín vậy.',
  'predicted_label': {'l1': 'sau_mua_hang',
   'l2': 'van_de_giao_hang',
   'l3': 'thieu_qua_tang_khuyen_mai'},
  'confidence': 0.9,
  'api_calls': 0,
  'input_tokens': 0,
  'output_tokens': 0,
  'cost_usd': 0.0,
  'metadata': {'source': 'd2_dataset_cache',
   'qa_status': 'approved',
   'verifier_decision': 'REVISE',
   'variant': 'cvhil_d',
   'use_retrieval': True,
   'enable_verifier': True,
   'enable_validation': True}},
 {'sample_id': 'nuoc-hoa-hong-khong-mui-klairs-danh-cho-da-nhay-cam-180ml-65994_q00',
  'sentence': 'Có thai xài sản phẩm này được ko shop?',
  'predicted_label': {'l1': 'truoc_mua_hang',
   'l2': 'an_toan_da',
   'l3': 'phu_hop_cho_nguoi_co_thai'},
  'confidence': 0.98,
  'api_calls': 0,
  'input_tokens': 0,
  'output_tokens': 0,
  'cost_u

## 6. Evaluate RQ2 ablation table

In [8]:
import subprocess

cmd = [
    sys.executable,
    str(REPO_ROOT / "experiments/annotation_benchmark/eval_rq2_ablation.py"),
    "--outdir", str(OUTDIR),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

md_path = OUTDIR / "ablation_quality_paper.md"
if md_path.exists():
    print("\n" + md_path.read_text(encoding="utf-8"))

/usr/bin/python3 /content/drive/MyDrive/intent_kb/experiments/annotation_benchmark/eval_rq2_ablation.py --outdir /content/drive/MyDrive/intent_kb/experiments/annotation_benchmark/results/rq2_ablation

| Variant | N | L1 Acc | L2 Acc | L3 Acc | L3 Macro-F1 | Full-path Acc |
| --- | --- | --- | --- | --- | --- | --- |
| A. LLM-only | 300 | 95.3% | 17.3% | 3.0% | 1.0% | 0.3% |
| B. Retrieval-Augmented | 300 | 96.7% | 55.7% | 34.7% | 22.7% | 30.0% |
| C. Retrieval + Verification | 300 | 94.3% | 58.3% | 34.0% | 23.0% | 31.3% |
| D. Full CV-HIL | 300 | 96.7% | 65.7% | 50.7% | 39.9% | 46.0% |

*RQ2 component ablation on D3 human gold. D (Full CV-HIL) reused from verified_d2 cache when available.*
## Incremental gains (Δ vs previous variant)

| Step | Δ L3 Macro-F1 | Δ Full-path Acc |
| --- | --- | --- |
| A. LLM-only → B. Retrieval-Augmented | 21.7% | 29.7% |
| B. Retrieval-Augmented → C. Retrieval + Verification | 0.3% | 1.3% |
| C. Retrieval + Verification → D. Full CV-HIL | 16.9% | 14.7% |

In [9]:
# Inline eval (không cần subprocess)
import pandas as pd

csv_path = OUTDIR / "ablation_quality.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df[[
        "method", "num_matched",
        "l1_accuracy", "l2_accuracy", "l3_accuracy",
        "l3_macro_f1", "full_path_accuracy",
    ]])
else:
    print("Chưa có", csv_path, "— chạy §5 hoặc §6 trước.")

,method,num_matched,l1_accuracy,l2_accuracy,l3_accuracy,l3_macro_f1,full_path_accuracy
0,A. LLM-only,300,0.953333,0.173333,0.030000,0.009599,0.003333
1,B. Retrieval-Augmented,300,0.966667,0.556667,0.346667,0.226697,0.300000
2,C. Retrieval + Verification,300,0.943333,0.583333,0.340000,0.229981,0.313333
3,D. Full CV-HIL,300,0.966667,0.656667,0.506667,0.398523,0.460000
